# NBA Win Probability — Colab Training Notebook

Trains an XGBoost win-probability model using pre-built PBP CSVs from `nba-on-court`.
Saves artifacts to Google Drive for use with `nba-bot-scan`.

**Run cells top-to-bottom on a fresh runtime.**

In [ ]:
# Cell 1 — Setup: install dependencies and nba_bot package
!pip install "nba-on-court>=0.3.0,<1.0" xgboost scikit-learn pandas numpy joblib tqdm nba_api -q

# FIXME: Replace YOUR_REPO/nba_bot with your actual GitHub repo URL before running
import os
if not os.path.exists('/content/nba_bot'):
    !git clone https://github.com/YOUR_REPO/nba_bot.git /content/nba_bot
!pip install -e /content/nba_bot --quiet

print('✅ Setup complete')

In [ ]:
# Cell 2 — Mount Google Drive (model artifacts will be saved here)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/nba_bot/'
import os; os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Output dir: {DRIVE_OUTPUT}')

In [ ]:
# Cell 3 — Config: choose seasons and feature tier
SEASONS = [2021, 2022, 2023, 2024]  # season start years
USE_ADVANCED_FEATURES = True         # True = Tier-2 model (better), False = Tier-1 (faster)

print(f'Seasons: {SEASONS}')
print(f'Tier: {"T2 (advanced)" if USE_ADVANCED_FEATURES else "T1 (baseline)"}')

In [ ]:
# Cell 4 — Download play-by-play data from shufinskiy/nba_data
# nba-on-court downloads pre-built CSV files — seconds vs hours with nba_api
import nba_on_court as noc

df_nbastats = noc.load_nba_data(seasons=SEASONS, data='nbastats')
print(f'nbastats: {len(df_nbastats):,} rows')

df_pbpstats = None
if USE_ADVANCED_FEATURES:
    df_pbpstats = noc.load_nba_data(seasons=SEASONS, data='pbpstats')
    print(f'pbpstats: {len(df_pbpstats):,} rows')

# Sanity check (AC 14: >= 200k rows for 2+ seasons)
assert len(df_nbastats) > 100_000, f'Expected >100k rows, got {len(df_nbastats)}'
print('✅ Data download OK')

In [ ]:
# Cell 5 — Fetch team net ratings (Tier-2 only)
# Single LeagueDashTeamStats call, all 30 teams. Used as player_quality feature.
# Note: fetches current-season ratings as proxy for all training seasons (known simplification).
player_ratings = {}
if USE_ADVANCED_FEATURES:
    from nba_bot.team_stats_cache import refresh_team_stats  # installed in Cell 1
    player_ratings = refresh_team_stats(output_path='/content/team_stats.json')
    print(f'Loaded ratings for {len(player_ratings)} teams')
    assert len(player_ratings) > 0, 'No team ratings returned — check NBA API'
else:
    print('Skipping team ratings (Tier-1 mode)')

In [ ]:
# Cell 6 — Feature engineering
from nba_bot.features import build_game_state_rows  # installed in Cell 1

df = build_game_state_rows(
    df_nbastats    = df_nbastats,
    df_pbpstats    = df_pbpstats,
    player_ratings = player_ratings,
    use_advanced   = USE_ADVANCED_FEATURES,
)

feature_tier = 'T2' if USE_ADVANCED_FEATURES else 'T1'
print(f'Dataset shape:   {df.shape}')
print(f'Class balance:   {df["home_win"].mean():.1%}  (should be ~50%)')
print(f'Feature tier:    {feature_tier}')
print(f'Games:           {df["game_id"].nunique():,}')
print()
print(df.head(3))

In [ ]:
# Cell 7 — Train models & evaluate
# Target: T1 AUC-ROC > 0.92 | T2 AUC-ROC > 0.93
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from nba_bot.config import FEATURES_T1, FEATURES_T2

feature_cols = FEATURES_T1 + (FEATURES_T2 if USE_ADVANCED_FEATURES else [])

df_train = df[df['time_remaining'] < 2870].dropna(subset=feature_cols + ['home_win'])
X = df_train[feature_cols].values
y = df_train['home_win'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# Logistic Regression baseline
lr = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]
print(f'LR  → log_loss={log_loss(y_test, lr_probs):.4f}  brier={brier_score_loss(y_test, lr_probs):.4f}  auc={roc_auc_score(y_test, lr_probs):.4f}')

# XGBoost main model
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, verbosity=0,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_probs)
print(f'XGB → log_loss={log_loss(y_test, xgb_probs):.4f}  brier={brier_score_loss(y_test, xgb_probs):.4f}  auc={xgb_auc:.4f}')

# AC 15 validation
min_auc = 0.91 if USE_ADVANCED_FEATURES else 0.90
assert xgb_auc > min_auc, f'AUC-ROC {xgb_auc:.4f} below threshold {min_auc}'
print(f'✅ AUC-ROC check passed ({xgb_auc:.4f} > {min_auc})')

In [ ]:
# Cell 8 — Save model artifacts to Google Drive
import joblib

model_name = f"xgb_model_t{'2' if USE_ADVANCED_FEATURES else '1'}.pkl"

# Save XGBoost model
joblib.dump(xgb_model, f'{DRIVE_OUTPUT}{model_name}')

# Save feature_cols.pkl — records the ordered feature list used at training time.
# model.py's load_model() reads this to validate feature order at inference (F12 fix).
joblib.dump(feature_cols, f'{DRIVE_OUTPUT}feature_cols.pkl')

print(f'✅ Saved: {DRIVE_OUTPUT}{model_name}')
print(f'✅ Saved: {DRIVE_OUTPUT}feature_cols.pkl')
print()
print('Download these files from Drive, then:')
print(f'  export NBA_BOT_MODEL_PATH=/path/to/{model_name}')
print('  nba-bot-scan --mode test')

In [ ]:
# Cell 9 — Inference smoke test
# nba_bot is installed in Cell 1, so this import will succeed (F4 fix)
from nba_bot.features import compute_features

model_check = joblib.load(f'{DRIVE_OUTPUT}{model_name}')
X_test_smoke = compute_features(home_score=88, away_score=84, period=4, clock='3:20')
prob = float(model_check.predict_proba(X_test_smoke)[0][1])

assert 0.4 < prob < 0.99, f'Suspicious output: {prob}  (expected 0.4–0.99 for this scenario)'
print(f'✅ Inference OK — home win prob: {prob:.3f}')
print(f'   (Lakers leading by 4 with 3:20 left in Q4)')